# 01 — Explore the original immune-health cohort

This notebook explores transformed marker values and gated cell populations without loading the full corpus. It samples a configurable number of files and cells per file, then provides marker distributions, correlations, population composition, PCA/UMAP, numerical checks, and rare-population retention estimates.

The source matrices are already preprocessed and arcsinh-transformed; they are not raw detector counts.

## 1. Setup and sampling controls

In [ ]:
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)

RANDOM_SEED = 42
SAMPLES_PER_COHORT = 12
CELLS_PER_SAMPLE = 5_000
MAXIMUM_PCA_CELLS = 50_000
RUN_UMAP = True


def find_project_root() -> Path:
    """Find the repository root from either the root or notebooks directory."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate the MAESTRO project root")


project_root = find_project_root()
source_root = project_root / "data/raw/preprocessed-cyto-sources"
cohort_directories = {
    "allof": source_root / "allof",
    "prepro": source_root / "prepro",
}

for cohort_directory in cohort_directories.values():
    if not cohort_directory.exists():
        raise FileNotFoundError(cohort_directory)

rng = np.random.default_rng(RANDOM_SEED)

## 2. Select files and sample cells

HDF5 requires sorted indices for efficient fancy indexing. Cells are sampled randomly, sorted for reading, and returned as a tidy table.

In [ ]:
def decode_strings(values: np.ndarray) -> list[str]:
    """Decode an HDF5 byte-string array."""
    return [
        value.decode("utf-8") if isinstance(value, bytes) else str(value)
        for value in values
    ]


def choose_files(
    directory: Path,
    number_files: int,
    generator: np.random.Generator,
) -> list[Path]:
    """Choose a reproducible random subset of HDF5 files."""
    files = sorted(directory.glob("*.h5"))
    selected_indices = generator.choice(
        len(files),
        size=min(number_files, len(files)),
        replace=False,
    )
    return [files[index] for index in sorted(selected_indices)]


def sample_hdf5_cells(
    file_path: Path,
    cohort: str,
    number_cells: int,
    generator: np.random.Generator,
) -> pd.DataFrame:
    """Sample cells and labels from one cytometry HDF5 file."""
    with h5py.File(file_path, "r") as h5_file:
        available_cells = h5_file["data"].shape[0]
        selected_indices = np.sort(
            generator.choice(
                available_cells,
                size=min(number_cells, available_cells),
                replace=False,
            )
        )
        marker_names = decode_strings(h5_file["feature_names"][:])
        marker_values = h5_file["data"][selected_indices]
        cell_types = decode_strings(h5_file["cell_types"][selected_indices])

    sampled = pd.DataFrame(marker_values, columns=marker_names)
    sampled.insert(0, "cell_type", cell_types)
    sampled.insert(0, "sample_name", file_path.stem)
    sampled.insert(0, "cohort_directory", cohort)
    return sampled


selected_files = {
    cohort: choose_files(directory, SAMPLES_PER_COHORT, rng)
    for cohort, directory in cohort_directories.items()
}
sampled_cells = pd.concat(
    [
        sample_hdf5_cells(file_path, cohort, CELLS_PER_SAMPLE, rng)
        for cohort, file_paths in selected_files.items()
        for file_path in file_paths
    ],
    ignore_index=True,
)
marker_names = [
    column
    for column in sampled_cells.columns
    if column not in {"cohort_directory", "sample_name", "cell_type"}
]

print(f"Sampled files: {sum(map(len, selected_files.values()))}")
print(f"Sampled cells: {len(sampled_cells):,}")
print(f"Markers: {len(marker_names)}")
sampled_cells.head()

## 3. Basic numerical quality checks

In [ ]:
marker_matrix = sampled_cells[marker_names].to_numpy()
quality_summary = pd.DataFrame(
    {
        "sampled_cells": [len(sampled_cells)],
        "finite_fraction": [np.isfinite(marker_matrix).mean()],
        "zero_fraction": [np.equal(marker_matrix, 0).mean()],
        "negative_fraction": [np.less(marker_matrix, 0).mean()],
        "minimum": [np.nanmin(marker_matrix)],
        "maximum": [np.nanmax(marker_matrix)],
    }
)
display(quality_summary)

marker_summary = (
    sampled_cells[marker_names].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T
)
marker_summary

## 4. Marker distributions and correlations

In [ ]:
markers_to_plot = [
    marker
    for marker in ["CD3", "CD4", "CD8a", "CD14", "CD19", "CD56", "CD66b", "HLA-DR"]
    if marker in marker_names
]
distribution_data = sampled_cells.melt(
    id_vars=["cohort_directory", "sample_name", "cell_type"],
    value_vars=markers_to_plot,
    var_name="marker",
    value_name="arcsinh_intensity",
)

plot = sns.displot(
    data=distribution_data,
    x="arcsinh_intensity",
    hue="cohort_directory",
    col="marker",
    col_wrap=4,
    kind="hist",
    stat="density",
    common_norm=False,
    element="step",
    bins=60,
    height=3,
    facet_kws={"sharex": False, "sharey": False},
)
plot.set_axis_labels("Arcsinh-transformed intensity", "Density")
plot.fig.suptitle("Selected marker distributions", y=1.02)
plt.show()

In [ ]:
correlation_matrix = sampled_cells[marker_names].corr(method="spearman")

plt.figure(figsize=(13, 11))
sns.heatmap(
    correlation_matrix,
    cmap="vlag",
    center=0,
    square=True,
    cbar_kws={"label": "Spearman correlation"},
)
plt.title("Marker correlation structure across sampled cells")
plt.tight_layout()
plt.show()

## 5. Gated population composition

Cell-type labels are auxiliary annotations and were not MAESTRO encoder inputs. They are useful here for confirming that the legacy snapshot contains the whole-blood granulocyte compartment.

In [ ]:
population_counts = (
    sampled_cells.groupby(
        ["cohort_directory", "sample_name", "cell_type"],
        observed=True,
    )
    .size()
    .rename("sampled_cells")
    .reset_index()
)
sampled_totals = population_counts.groupby(
    ["cohort_directory", "sample_name"], observed=True
)["sampled_cells"].transform("sum")
population_counts["sampled_fraction"] = (
    population_counts["sampled_cells"] / sampled_totals
)

population_order = (
    population_counts.groupby("cell_type", observed=True)["sampled_cells"]
    .sum()
    .sort_values(ascending=False)
    .index
)

plt.figure(figsize=(12, 7))
sns.boxplot(
    data=population_counts,
    y="cell_type",
    x="sampled_fraction",
    hue="cohort_directory",
    order=population_order,
    showfliers=False,
)
plt.xlabel("Fraction of sampled cells")
plt.ylabel("")
plt.title("Cell-population composition across sampled files")
plt.tight_layout()
plt.show()

In [ ]:
population_marker_medians = sampled_cells.groupby("cell_type", observed=True)[
    marker_names
].median()
population_marker_z_scores = population_marker_medians.apply(
    lambda values: (values - values.mean()) / values.std(ddof=0),
    axis=0,
).fillna(0)

plt.figure(figsize=(15, 9))
sns.heatmap(
    population_marker_z_scores,
    cmap="vlag",
    center=0,
    cbar_kws={"label": "Marker-wise z-score of median intensity"},
)
plt.title("Canonical marker profiles of gated populations")
plt.xlabel("Marker")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 6. PCA of sampled cells

This projection visualizes individual cells rather than MAESTRO sample embeddings.

In [ ]:
projection_indices = rng.choice(
    len(sampled_cells),
    size=min(MAXIMUM_PCA_CELLS, len(sampled_cells)),
    replace=False,
)
projection_cells = sampled_cells.iloc[projection_indices].reset_index(drop=True)
scaled_markers = StandardScaler().fit_transform(projection_cells[marker_names])
principal_components = PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(
    scaled_markers
)
projection_cells["PC1"] = principal_components[:, 0]
projection_cells["PC2"] = principal_components[:, 1]

figure, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.scatterplot(
    data=projection_cells,
    x="PC1",
    y="PC2",
    hue="cohort_directory",
    alpha=0.25,
    s=8,
    linewidth=0,
    ax=axes[0],
)
axes[0].set_title("PCA colored by physical cohort directory")

sns.scatterplot(
    data=projection_cells,
    x="PC1",
    y="PC2",
    hue="cell_type",
    alpha=0.25,
    s=8,
    linewidth=0,
    ax=axes[1],
)
axes[1].set_title("PCA colored by gated population")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)

plt.tight_layout()
plt.show()

## 7. Optional UMAP

Set `RUN_UMAP = True` in the first cell to run this slower projection.

In [ ]:
if RUN_UMAP:
    from umap import UMAP

    umap_coordinates = UMAP(
        n_neighbors=30,
        min_dist=0.2,
        metric="euclidean",
        random_state=RANDOM_SEED,
    ).fit_transform(scaled_markers)
    projection_cells["UMAP1"] = umap_coordinates[:, 0]
    projection_cells["UMAP2"] = umap_coordinates[:, 1]

    plt.figure(figsize=(11, 8))
    sns.scatterplot(
        data=projection_cells,
        x="UMAP1",
        y="UMAP2",
        hue="cell_type",
        alpha=0.3,
        s=8,
        linewidth=0,
    )
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
    plt.title("UMAP of sampled cells")
    plt.tight_layout()
    plt.show()
else:
    print("UMAP skipped. Set RUN_UMAP = True to enable it.")

## 8. Rare-population retention under cell subsampling

For a population with sample fraction $p$ and a random subsample of $M$ cells, the expected count is $Mp$ and the probability of retaining at least one cell is $1-(1-p)^M$.

In [ ]:
SUBSAMPLE_SIZES = [5_000, 40_000, 100_000]
median_population_fractions = (
    population_counts.groupby("cell_type", observed=True)["sampled_fraction"]
    .median()
    .sort_values()
)

retention_rows = []
for cell_type, population_fraction in median_population_fractions.items():
    for subsample_size in SUBSAMPLE_SIZES:
        retention_rows.append(
            {
                "cell_type": cell_type,
                "median_sampled_fraction": population_fraction,
                "subsample_size": subsample_size,
                "expected_cells": subsample_size * population_fraction,
                "probability_at_least_one": 1
                - (1 - population_fraction) ** subsample_size,
            }
        )

retention_table = pd.DataFrame(retention_rows)
retention_table.pivot(
    index=["cell_type", "median_sampled_fraction"],
    columns="subsample_size",
    values="expected_cells",
).rename_axis(columns="Expected cells at subsample size")

## Interpretation checklist before training

- Confirm that the sampled files share the expected 30-marker panel.
- Decide whether to reproduce the paper's inclusion of low-quality samples or train a cleaner `QC == PASS` sensitivity model.
- Preserve whole-blood granulocytes for the closest approximation to the original MAESTRO corpus.
- Split longitudinal specimens by `patient_id`, never independently by file.
- Quantify how cell caps affect rare populations before changing the original sampling strategy.
- Freeze an explicit manifest because the available 1,870-file snapshot is not the paper's exact 1,792-file manifest.